# Foresight: Model Development
Exploration and prototyping for trajectory prediction models.

In [ ]:
import sys
sys.path.insert(0, '../backend')

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.style.use('dark_background')
ACCENT = '#22d3ee'
TARGET_COLOR = '#a3e635'
PRED_COLOR = '#f97316'

In [ ]:
# Load preprocessed data
from pathlib import Path

data_dir = Path('../backend/app/data/processed')
train_data = torch.load(data_dir / 'train.pt', weights_only=True)
val_data = torch.load(data_dir / 'val.pt', weights_only=True)
test_data = torch.load(data_dir / 'test.pt', weights_only=True)

print(f'Train: {train_data["X"].shape}')
print(f'Val:   {val_data["X"].shape}')
print(f'Test:  {test_data["X"].shape}')

In [ ]:
# Visualize sample trajectories
fig, axes = plt.subplots(2, 4, figsize=(16, 8), facecolor='#0a0a0a')
axes = axes.flatten()

indices = np.random.choice(len(train_data['X']), 8, replace=False)

for ax, idx in zip(axes, indices):
    x = train_data['X'][idx].numpy()   # (20, 4)
    y = train_data['y'][idx].numpy()   # (20, 2)

    ax.set_facecolor('#080e18')
    ax.plot(x[:, 0], x[:, 1], color=ACCENT, linewidth=2, label='Input')
    ax.plot(y[:, 0], y[:, 1], color=TARGET_COLOR, linewidth=2, label='Target')
    ax.scatter([x[-1, 0]], [x[-1, 1]], color=ACCENT, s=40, zorder=5)
    ax.scatter([y[-1, 0]], [y[-1, 1]], color=TARGET_COLOR, s=40, zorder=5)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f'Sample {idx}', color='#737373', fontsize=10)

handles = [mpatches.Patch(color=ACCENT, label='Input'), mpatches.Patch(color=TARGET_COLOR, label='Target')]
fig.legend(handles=handles, loc='lower center', ncol=2, facecolor='#111', edgecolor='#252525')
plt.suptitle('Training Trajectory Samples', color='#f0f0f0', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Inspect velocity distribution
dx = train_data['X'][:, :, 2].numpy().flatten()
dy = train_data['X'][:, :, 3].numpy().flatten()

fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor='#0a0a0a')
for ax, vals, label in zip(axes, [dx, dy], ['dx (horizontal velocity)', 'dy (vertical velocity)']):
    ax.set_facecolor('#111')
    ax.hist(vals, bins=80, color=ACCENT, alpha=0.7, edgecolor='none')
    ax.set_title(label, color='#f0f0f0')
    ax.set_xlabel('Normalized velocity', color='#737373')
    ax.tick_params(colors='#737373')
    for spine in ax.spines.values():
        spine.set_edgecolor('#252525')
plt.tight_layout()
plt.show()

In [ ]:
# Load training history and plot
import json

history_path = Path('../backend/app/model/training_history.json')
if history_path.exists():
    history = json.loads(history_path.read_text())

    fig, ax = plt.subplots(figsize=(10, 5), facecolor='#0a0a0a')
    ax.set_facecolor('#111')
    ax.plot(history['epochs'], history['train_loss'], color=ACCENT, linewidth=2, label='Train loss')
    ax.plot(history['epochs'], history['val_loss'], color=PRED_COLOR, linewidth=2, label='Val loss')
    ax.set_xlabel('Epoch', color='#737373')
    ax.set_ylabel('Huber Loss', color='#737373')
    ax.set_title(f"Training History ({history['model']})", color='#f0f0f0')
    ax.legend(facecolor='#111', edgecolor='#252525', labelcolor='#f0f0f0')
    ax.tick_params(colors='#737373')
    for spine in ax.spines.values():
        spine.set_edgecolor('#252525')
    plt.tight_layout()
    plt.show()
    print(f'Best val loss: {history["best_val_loss"]}')
else:
    print('No training history found. Run training first.')

In [ ]:
# Run inference with MC dropout uncertainty
from app.model.predict import predict, model_is_trained

if model_is_trained('transformer'):
    idx = np.random.randint(len(test_data['X']))
    x = test_data['X'][idx].numpy()
    y = test_data['y'][idx].numpy()

    result = predict(x, model_name='transformer', mc_passes=20)
    mean = np.array(result['mean'])
    std = np.array(result['std'])
    samples = np.array(result['samples'])

    fig, ax = plt.subplots(figsize=(8, 7), facecolor='#0a0a0a')
    ax.set_facecolor('#080e18')

    # MC samples (faint)
    for sample in samples:
        ax.plot(sample[:, 0], sample[:, 1], color=PRED_COLOR, alpha=0.08, linewidth=1)

    # Input, target, mean prediction
    ax.plot(x[:, 0], x[:, 1], color=ACCENT, linewidth=2.5, label='Input')
    ax.plot(y[:, 0], y[:, 1], color=TARGET_COLOR, linewidth=2.5, label='Ground truth')
    ax.plot(mean[:, 0], mean[:, 1], color=PRED_COLOR, linewidth=2.5, label='Prediction (mean)')

    # Uncertainty envelope
    for i, (mx, my) in enumerate(mean):
        sx = (std[i, 0] + std[i, 1]) / 2
        circle = plt.Circle((mx, my), sx * 5, color=PRED_COLOR, alpha=0.12)
        ax.add_patch(circle)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title(f'MC Dropout Prediction (sample {idx})', color='#f0f0f0')
    ax.legend(facecolor='#111', edgecolor='#252525', labelcolor='#f0f0f0')
    ax.tick_params(colors='#737373')
    for spine in ax.spines.values():
        spine.set_edgecolor('#252525')
    plt.tight_layout()
    plt.show()

    ade = float(np.mean(np.linalg.norm(mean - y, axis=1)))
    fde = float(np.linalg.norm(mean[-1] - y[-1]))
    print(f'ADE: {ade:.5f}')
    print(f'FDE: {fde:.5f}')
else:
    print('Transformer model not trained yet.')